# Validation Set

There is no public labeled dataset of warehouses or trailers on NJ NAIP imagery, so instead of hand-labeling hundreds of training chips, this notebook builds a small set for checking results by eye:

1. **NAIP resolution per year**: confirm which years are 1 m, 60 cm or 30 cm (it decides which years can show trailers).
2. **Warehouse inventory**: rule-based, no model. Footprints on industrial parcels (MOD-IV class `4B`) of at least 50,000 sq ft.
3. **Year built**: how much of the inventory MOD-IV can date.
4. **Precision sample**: 100 random inventory buildings, each to be marked warehouse / not warehouse.
5. **Recall cells**: 20 random 1 km cells, scanned for warehouses the inventory missed.
6. **Dock-yard sites**: ~40 large warehouses with truck yards, spread across the corridor and NAIP years, for counting trailers by eye (keep ~30 busy ones). These are the ground truth for the trailer detector.

Images are written under `data/interim/chips/{precision,recall,sites}/` and labeled in Label Studio (`labeling/LABELING_GUIDE.md`).

Input: `data/interim/warehouse_candidates.gpkg` (from `notebooks/02_warehouse_candidates.ipynb`)

### Libraries

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from PIL import Image, ImageDraw
from rasterio.merge import merge
from shapely.geometry import box

DATA = Path("/Volumes/01-2TB/NJ-Truck-and-Warhouse-Pollution/data")  # external SSD
RAW = DATA / "raw"
INTERIM = DATA / "interim"
CHIP_DIR = INTERIM / "chips"  # Label Studio serves images from here
NAIP_DIR = RAW / "naip"
UTM = "EPSG:26918"            # NAIP tiles are in UTM zone 18N (NAD83), metres

SEED = 42
LARGE_M2 = 4_645              # inventory threshold (= 50,000 sq ft)
N_PRECISION = 100             # inventory buildings checked for precision
PRECISION_YEAR = 2022         # newest 60 cm year
N_RECALL_CELLS = {"large_building": 12, "other": 8}   # 1 km cells per stratum
CELL_M = 1_000                # recall cells match the 1 km air-quality grid
SITE_MIN_M2 = 9_290           # dock-yard sites: at least 100,000 sq ft, big enough for a real truck yard
SITE_QUOTA = {"Middlesex": 24, "Union": 8, "Essex": 8}   # ~40 sites; keep ~30 busy ones after screening
YARD_MARGIN_M = 80            # site window = footprint bounds + this margin, same ground area in every year
EXTRA_TILES = {2019: 8, 2015: 6, 2010: 6, 2023: 10}   # extra site tiles per year, on top of one 2022 tile per site

rng = np.random.default_rng(SEED)

### Load candidates and NAIP tiles

In [ ]:
assert NAIP_DIR.exists(), f"{NAIP_DIR} not found; plug in the external SSD"

candidates = gpd.read_file(INTERIM / "warehouse_candidates.gpkg", layer="candidates").to_crs(UTM)

naip_all = gpd.read_file(NAIP_DIR / "naip_index.geojson").to_crs(UTM)
naip_all["downloaded"] = naip_all["path"].map(lambda p: (NAIP_DIR / p).exists())
naip = naip_all[naip_all["downloaded"]].copy()
naip["block"] = naip["id"].str.extract(r"m_(\d{7}_[a-z]{2})", expand=False)  # quarter-quad, same in every year
GSD = naip.groupby("year")["gsd"].first().to_dict()

study_area = gpd.read_file(RAW / "core_counties.geojson").to_crs(UTM).union_all()
print(f"{len(candidates):,} candidates; downloaded NAIP years {GSD}")

### 1. NAIP resolution per year
`gsd_index` is the resolution listed in the tile index (every year in the study area). For downloaded years, `gsd_file_min`/`max` is read from every file's header, to confirm it.

In [ ]:
def file_res(path):
    with rasterio.open(NAIP_DIR / path) as src:
        return src.res[0]


naip["gsd_file"] = naip["path"].map(file_res)
resolution = (naip_all.groupby("year").agg(tiles=("id", "size"), downloaded=("downloaded", "sum"), gsd_index=("gsd", "first"))
              .join(naip.groupby("year")["gsd_file"].agg(["min", "max"]).add_prefix("gsd_file_")))
resolution

A semi-trailer is about 16 × 2.6 m: roughly 16 × 3 px at 1 m, 27 × 4 px at 60 cm, and 53 × 9 px at 30 cm. Trailer counts in 1 m years will be rough, so the site tiles below include 1 m, 60 cm and 30 cm years, to measure how accuracy changes with resolution.

### 2. Warehouse inventory
Rule: a candidate footprint (Microsoft, plus OSM-only additions) whose roof point falls on a MOD-IV class `4B` (industrial) parcel and whose area is at least 4,645 m² (50,000 sq ft). `block` is the NAIP quarter-quad containing its centre; it's used to spread samples out.

In [ ]:
inventory = candidates[(candidates["prop_class"] == "4B") & (candidates["area_m2"] >= LARGE_M2)].copy()

tiles_2022 = naip.loc[naip["year"] == 2022, ["block", "geometry"]]
centres = gpd.GeoDataFrame(inventory[["cand_id"]], geometry=inventory.centroid, crs=UTM)
block_of = gpd.sjoin(centres, tiles_2022, predicate="within").drop_duplicates("cand_id").set_index("cand_id")["block"]
inventory["block"] = inventory["cand_id"].map(block_of)

print(f"Inventory: {len(inventory):,} buildings, {inventory['area_m2'].sum() / 1e6:.1f} km² of roof")
pd.crosstab(inventory["county"].fillna("outside"), inventory["source"], margins=True)

### 3. Year built (MOD-IV `YR_CONSTR`)
A first guess at when each warehouse appeared. Two caveats: it describes the parcel's main building, which may not be this footprint on parcels with several buildings, and `0` means not recorded.

In [ ]:
yr = pd.to_numeric(inventory["yr_constr"], errors="coerce")
inventory["year_built"] = yr.where(yr.between(1800, 2026))

has_year = inventory["year_built"].notna()
print(f"Usable year built: {has_year.sum():,} of {len(inventory):,} ({has_year.mean():.0%})")
print(f"0 or blank: {((yr == 0) | yr.isna()).sum():,}; other invalid (e.g. 9999): {(yr.notna() & (yr != 0) & ~has_year).sum():,}")

bins = [1799, 1980, 2000, 2010, 2015, 2019, 2022, 2026]
labels = ["before 1981", "1981-2000", "2001-2010", "2011-2015", "2016-2019", "2020-2022", "2023+"]
pd.cut(inventory["year_built"], bins, labels=labels).value_counts().sort_index().rename("buildings").to_frame()

Under half the inventory has a usable year, so the per-year "roof present yet?" check (Month 3) is still needed; `year_built` is a starting point and a cross-check for it.

In [ ]:
inventory.to_file(INTERIM / "warehouse_inventory.gpkg", layer="inventory", driver="GPKG")
print(f"Saved {len(inventory):,} buildings to data/interim/warehouse_inventory.gpkg")

### Image export
`export_image` cuts an RGB PNG from every NAIP tile of one year that touches a window (merged across tile edges), optionally outlines footprints in yellow, and returns the affine transform (`a`–`f`) that maps pixel (col, row) to UTM: `x = a·col + b·row + c`, `y = d·col + e·row + f`. Label Studio clicks can be converted back to map coordinates with it.

In [ ]:
def square_box(geom, margin, min_side=200):
    """Square window centred on geom, covering it plus a margin, at least min_side metres across."""
    minx, miny, maxx, maxy = geom.bounds
    side = max(maxx - minx, maxy - miny, min_side - 2 * margin) + 2 * margin
    cx, cy = (minx + maxx) / 2, (miny + maxy) / 2
    return box(cx - side / 2, cy - side / 2, cx + side / 2, cy + side / 2)


def export_image(window, year, out, outlines=()):
    tiles = naip[(naip["year"] == year) & naip.intersects(window)]["path"]
    sources = [rasterio.open(NAIP_DIR / p) for p in tiles]
    try:
        rgb, transform = merge(sources, bounds=window.bounds, res=GSD[year], indexes=[1, 2, 3], nodata=0)
    finally:
        for src in sources:
            src.close()
    img = Image.fromarray(np.moveaxis(rgb, 0, -1))
    draw = ImageDraw.Draw(img)
    for geom in outlines:
        for poly in getattr(geom, "geoms", [geom]):
            draw.line([((x - transform.c) / transform.a, (y - transform.f) / transform.e) for x, y in poly.exterior.coords],
                      fill=(255, 230, 0), width=2)
    out.parent.mkdir(parents=True, exist_ok=True)
    img.save(out)
    return {"file": str(out.relative_to(CHIP_DIR)), "width": img.width, "height": img.height,
            **dict(zip("abcdef", tuple(transform)[:6]))}

### 4. Precision sample
100 random inventory buildings on the 2022 imagery, each outlined in yellow with ~40 m of surroundings. Each one gets marked warehouse / not warehouse. Precision = share marked warehouse.

In [ ]:
precision = inventory.sample(N_PRECISION, random_state=SEED)

records = [export_image(square_box(b.geometry, 40), PRECISION_YEAR, CHIP_DIR / "precision" / f"{b.cand_id}.png", [b.geometry])
           for b in precision.itertuples()]
precision_index = pd.concat([precision[["cand_id", "area_m2", "source", "county", "mun_name", "year_built"]].reset_index(drop=True),
                             pd.DataFrame(records)], axis=1).assign(year=PRECISION_YEAR, gsd=GSD[PRECISION_YEAR])
precision_index.to_csv(CHIP_DIR / "precision" / "index.csv", index=False)
print(f"Exported {len(precision_index)} precision images")
precision_index["county"].value_counts().to_frame()

### 5. Recall cells
A 1 km grid over the study area (cells fully covered by 2022 imagery). Purely random cells would mostly be housing with no warehouses, so cells are split into two strata and sampled separately:
- **large_building**: contains the centre of any candidate building of at least 50,000 sq ft, whatever its parcel class (where warehouses are, without assuming the `4B` rule is right)
- **other**: everything else (catches warehouses missing from both footprint datasets)

Each cell is exported at 2022 with every inventory building outlined. The labeler clicks each warehouse that is **not** outlined (missed) and each outlined building that is **not** a warehouse. Recall per stratum = (outlined − not warehouse) / (outlined − not warehouse + missed); weight the two strata by their cell counts for an overall figure.

In [ ]:
minx, miny, maxx, maxy = study_area.bounds
xs = np.arange(minx // CELL_M * CELL_M, maxx, CELL_M)
ys = np.arange(miny // CELL_M * CELL_M, maxy, CELL_M)
cells = gpd.GeoDataFrame(geometry=[box(x, y, x + CELL_M, y + CELL_M) for x in xs for y in ys], crs=UTM)
cells = cells[cells.centroid.within(study_area) & cells.within(tiles_2022.union_all())].reset_index(drop=True)

def count_in_cells(buildings):
    centres = gpd.GeoDataFrame(geometry=buildings.centroid.values, crs=UTM)
    return gpd.sjoin(cells, centres, predicate="contains").groupby(level=0).size().reindex(cells.index, fill_value=0)


cells["n_large"] = count_in_cells(candidates[candidates["is_large"]])
cells["n_inventory"] = count_in_cells(inventory)
cells["stratum"] = np.where(cells["n_large"] > 0, "large_building", "other")
print(cells.groupby("stratum").agg(cells=("stratum", "size"), large_buildings=("n_large", "sum"),
                                   inventory_buildings=("n_inventory", "sum")).to_string())

recall = pd.concat([cells[cells["stratum"] == s].sample(n, random_state=SEED) for s, n in N_RECALL_CELLS.items()])
recall.insert(0, "cell_id", [f"r{i:02d}" for i in range(1, len(recall) + 1)])

records = []
for cell in recall.itertuples():
    shown = inventory[inventory.intersects(cell.geometry)].geometry
    records.append(export_image(cell.geometry, PRECISION_YEAR, CHIP_DIR / "recall" / f"{cell.cell_id}.png", shown))
recall_index = pd.concat([recall.drop(columns="geometry").reset_index(drop=True), pd.DataFrame(records)], axis=1)
recall_index = recall_index.assign(year=PRECISION_YEAR, gsd=GSD[PRECISION_YEAR])
recall_index.to_csv(CHIP_DIR / "recall" / "index.csv", index=False)
recall_index.groupby("stratum").agg(cells=("cell_id", "size"), large_buildings=("n_large", "sum"),
                                    inventory_buildings=("n_inventory", "sum"))

### 6. Dock-yard sites
Ground truth for trailer counts. Sites are inventory warehouses of at least 100,000 sq ft, picked at random per county (Middlesex has most of the inventory, so most sites), with at most 2 per NAIP block and no two site windows overlapping, so no trailer is counted twice.

Each site's window (footprint + 80 m) is the same ground area in every year. Every site gets a 2022 tile; 30 sites also get one extra tile: an older one (2019, 2015 or 2010), chosen among sites whose MOD-IV year built is unknown or no later than that year, or a 2023 one (30 cm), which pairs with the 2022 tile to compare 30 cm and 60 cm counts on nearly the same yard. That gives 70 tiles; sites that turn out to have no truck yard are skipped while labeling, leaving ~30 sites and ~50 tiles.

Labeling = one click per trailer, so the count is the number of clicks, and clicks can later be matched to detections one by one. These sites' blocks must stay out of any detector fine-tuning tiles.

In [ ]:
pool = inventory[inventory["area_m2"] >= SITE_MIN_M2]
print(f"Warehouses >= {SITE_MIN_M2:,} m2: {len(pool):,}; by county {pool['county'].value_counts().to_dict()}")

picked, windows, per_block = [], [], {}
for county, n in SITE_QUOTA.items():
    n_county = 0
    for b in pool[pool["county"] == county].sample(frac=1, random_state=SEED).itertuples():
        minx, miny, maxx, maxy = b.geometry.bounds
        win = box(minx - YARD_MARGIN_M, miny - YARD_MARGIN_M, maxx + YARD_MARGIN_M, maxy + YARD_MARGIN_M)
        if per_block.get(b.block, 0) >= 2 or any(win.intersects(w) for w in windows):
            continue
        picked.append({"cand_id": b.cand_id, "county": county, "block": b.block, "area_m2": b.area_m2,
                       "year_built": b.year_built, "geometry": win})
        windows.append(win)
        per_block[b.block] = per_block.get(b.block, 0) + 1
        n_county += 1
        if n_county == n:
            break

sites = gpd.GeoDataFrame(picked, crs=UTM)
sites.insert(0, "site_id", [f"s{i:02d}" for i in range(1, len(sites) + 1)])

tile_rows, available = [], set(sites["site_id"])
for site in sites.itertuples():
    tile_rows.append({"site_id": site.site_id, "year": 2022})
tiles_per_year = naip_all.groupby("year")["downloaded"].agg(["size", "sum"])
for year, n in EXTRA_TILES.items():
    assert tiles_per_year.loc[year, "sum"] == tiles_per_year.loc[year, "size"], f"NAIP {year} not fully downloaded"
    covered = sites.within(naip.loc[naip["year"] == year].union_all())
    eligible = sites[sites["site_id"].isin(available) & covered & (sites["year_built"].isna() | (sites["year_built"] <= year))]
    for site_id in eligible["site_id"].sample(min(n, len(eligible)), random_state=SEED):
        tile_rows.append({"site_id": site_id, "year": year})
        available.discard(site_id)
site_tiles = pd.DataFrame(tile_rows).merge(sites, on="site_id")

records = []
for t in site_tiles.itertuples():
    records.append(export_image(t.geometry, t.year, CHIP_DIR / "sites" / f"{t.year}_{t.site_id}.png"))
site_index = pd.concat([site_tiles.drop(columns="geometry"), pd.DataFrame(records)], axis=1)
site_index["gsd"] = site_index["year"].map(GSD)
site_index.to_csv(CHIP_DIR / "sites" / "index.csv", index=False)
print(f"{len(sites)} sites, {len(site_index)} tiles")
pd.crosstab(site_index["county"], site_index["year"], margins=True)

### Save sample locations
One GeoPackage with the three samples, for mapping and for keeping these places out of any training data.

In [ ]:
out = INTERIM / "validation.gpkg"
precision[["cand_id", "area_m2", "county", "block", "geometry"]].to_file(out, layer="precision", driver="GPKG")
recall[["cell_id", "stratum", "n_large", "n_inventory", "geometry"]].to_file(out, layer="recall_cells", driver="GPKG")
sites.to_file(out, layer="sites", driver="GPKG")
print(f"Saved precision, recall_cells and sites layers to data/interim/validation.gpkg")

### Preview

In [ ]:
fig, ax = plt.subplots(figsize=(8, 9))
gpd.GeoSeries([study_area], crs=UTM).boundary.plot(ax=ax, color="grey", linewidth=0.6)
inventory.centroid.plot(ax=ax, color="lightgrey", markersize=2, label="inventory")
recall.boundary.plot(ax=ax, color="tab:blue", linewidth=1.5)
precision.centroid.plot(ax=ax, color="tab:orange", markersize=10, label="precision sample")
sites.boundary.plot(ax=ax, color="tab:red", linewidth=1.5)
ax.plot([], [], color="tab:blue", label="recall cells")
ax.plot([], [], color="tab:red", label="dock-yard sites")
ax.legend(loc="lower right")
ax.set_axis_off()

examples = [CHIP_DIR / precision_index["file"].iloc[0], CHIP_DIR / recall_index["file"].iloc[0], CHIP_DIR / site_index["file"].iloc[0]]
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, path in zip(axes, examples):
    ax.imshow(Image.open(path))
    ax.set_title(path.relative_to(CHIP_DIR), fontsize=9)
    ax.set_axis_off()
plt.tight_layout()

### Next
1. `./labeling/start_label_studio.sh`, then `.venv-labelstudio/bin/python labeling/setup_projects.py` (creates the three validation projects).
2. Label following `labeling/LABELING_GUIDE.md`: precision (~1 evening), recall cells and sites (~1 evening).
3. Export each project as JSON to `data/interim/labels/`.